# Answer spans in a long passage

Fine-tuned for reading comprehension, BERT answers a question by pointing at a span of the
passage. Its input is `[CLS]`, the question, `[SEP]`, the passage, `[SEP]`. A linear layer
with two outputs on every position's vector gives each position a start score and an end
score. The answer is the span of passage pieces from position $i$ to position $j$ with the
largest $\text{start}_i + \text{end}_j$, where $i \le j$ and the span has at most $L$
pieces.

BERT reads at most 512 positions, and the inputs here are shorter still, so a long
passage is cut into windows that overlap. Each window is paired with the question and
read on its own, and the best span over all windows is the answer.

You will build one input, find the best span in it, and then answer over a whole passage.
A small stand-in reader replaces BERT, so everything runs in a second.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. `toy_reader` stands in for BERT and its span layer. Like a real
model's raw scores, its scores cover every position, including the question, the special
tokens and the padding, so picking a valid span is up to you.

In [ ]:
import numpy as np

PAD_ID, CLS_ID, SEP_ID = 0, 101, 102
STRONG = {5001: (9.0, -5.0), 5002: (-5.0, 9.0), 5003: (8.5, -5.0), 5004: (-5.0, 8.5)}


def toy_reader(input_ids, segment_ids, attention_mask):
    """A start score and an end score for every position, from the piece at that position."""
    ids = np.asarray(input_ids, dtype=np.int64)
    start = (ids * 7919 % 1009) / 100 - 5
    end = (ids * 6007 % 1013) / 100 - 5
    for piece, (s, e) in STRONG.items():
        start = np.where(ids == piece, s, start)
        end = np.where(ids == piece, e, end)
    return start, end


question = 2000 + np.arange(9)                       # 9 question pieces
passage_a = 3000 + np.arange(300) * 53 % 997         # 300 distinct passage pieces
passage_a[[290, 292]] = [5001, 5002]
passage_b = 3000 + np.arange(300) * 61 % 991
passage_b[[106, 109]] = [5003, 5004]
print(len(question), "question pieces,", len(passage_a), "and", len(passage_b), "passage pieces")

## Part 1 (coding task): One question and passage as BERT's input

Write `pair_input(question, passage, max_len)`. It returns three integer arrays of length
`max_len`:

- `input_ids`: `[CLS]`, the question, `[SEP]`, the passage, `[SEP]`, then `[PAD]` up to
  `max_len`. If the passage does not fit, keep only as many of its first pieces as fit,
  and keep the question whole.
- `segment_ids`: 0 for `[CLS]`, the question and the first `[SEP]`, 1 for the passage
  pieces and the last `[SEP]`, and 0 for the padding.
- `attention_mask`: 1 at every real position and 0 at the padding.

You may assume the question leaves room for at least one passage piece.

In [ ]:
def pair_input(question, passage, max_len):
    """input_ids, segment_ids and attention_mask for [CLS] question [SEP] passage [SEP], padded."""
    question = np.asarray(question)
    passage = np.asarray(passage)
    # TODO
    raise NotImplementedError

In [ ]:
ids, seg, mask = (np.asarray(a) for a in pair_input([2001, 2002], [3001, 3002, 3003], 10))
assert np.array_equal(ids, [101, 2001, 2002, 102, 3001, 3002, 3003, 102, 0, 0]), f"input_ids {ids.tolist()}"
assert np.array_equal(seg, [0, 0, 0, 0, 1, 1, 1, 1, 0, 0]), (
    f"segment_ids {seg.tolist()}: the first [SEP] is in segment 0, the last [SEP] in segment 1, padding 0"
)
assert np.array_equal(mask, [1, 1, 1, 1, 1, 1, 1, 1, 0, 0]), f"attention_mask {mask.tolist()}"
ids, seg, mask = (np.asarray(a) for a in pair_input([2001, 2002, 2003], 3000 + np.arange(20), 12))
assert len(ids) == len(seg) == len(mask) == 12, "every array must have length max_len"
assert np.array_equal(ids, [101, 2001, 2002, 2003, 102, 3000, 3001, 3002, 3003, 3004, 3005, 102]), (
    f"input_ids {ids.tolist()}: a passage that does not fit loses its last pieces, and the input still ends with [SEP]"
)
assert np.array_equal(seg, [0] * 5 + [1] * 7) and mask.sum() == 12, f"segment_ids {seg.tolist()}"
print("Part 1 passed")

## Part 2 (coding task): The best span in one input

Write `best_span(start, end, input_ids, segment_ids, max_answer_len)`. A span $(i, j)$ is
valid when both $i$ and $j$ are passage positions (segment 1 and not `[SEP]`), $i \le j$,
and it has at most `max_answer_len` pieces, so $j - i + 1 \le L$. Return `(i, j, score)`
for the valid span with the largest score $\text{start}_i + \text{end}_j$. If two
spans share that largest score, take the one with the smallest $i$, then the smallest
$j$.

Any correct method passes. Scoring every pair at once with broadcasting,
`start[:, None] + end[None, :]`, and setting invalid pairs to $-\infty$ is one way.

In [ ]:
def best_span(start, end, input_ids, segment_ids, max_answer_len):
    """(i, j, score) of the best valid span: passage positions only, i <= j, at most max_answer_len pieces."""
    start = np.asarray(start, dtype=float)
    end = np.asarray(end, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
ids1 = [101, 2001, 2002, 2003, 102, 3001, 3002, 3003, 3004, 3005, 3006, 3007, 102, 0]
seg1 = [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0]
s1 = [1.0, 2.0, 6.0, 1.5, 0.5, 3.0, 0.5, 1.0, 1.0, 4.0, 0.0, 0.5, 2.0, 5.0]
e1 = [0.5, 1.0, 2.0, 0.0, 1.0, 0.0, 3.5, 1.0, 1.5, 0.5, 2.5, 1.0, 6.0, 7.0]
got = best_span(s1, e1, ids1, seg1, 4)
assert tuple(got[:2]) == (5, 6) and abs(got[2] - 6.5) < 1e-9, (
    f"got {got}, expected (5, 6, 6.5): both ends must be passage pieces, the end cannot come before the start, and the smallest i wins a tie"
)
ids2 = [101, 2001, 2002, 102, 3001, 3002, 3003, 3004, 3005, 3006, 3007, 3008, 102, 0, 0]
seg2 = [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]
s2 = np.zeros(15); s2[[5, 9]] = [5.0, 3.5]
e2 = np.zeros(15); e2[[9, 10]] = [3.0, 4.0]
got = best_span(s2, e2, ids2, seg2, 5)
assert tuple(got[:2]) == (5, 9) and abs(got[2] - 8.0) < 1e-9, (
    f"got {got}, expected (5, 9, 8.0): a span from 5 to 9 has 5 pieces, which max_answer_len = 5 allows, and 5 to 10 has 6"
)
ids3 = [101, 2001, 102, 3001, 3002, 3003, 3004, 102]
seg3 = [0, 0, 0, 1, 1, 1, 1, 1]
got = best_span([0, 0, 0, 1, 0, 4, 0, 0], [0, 0, 0, 0, 2, 4, 1, 0], ids3, seg3, 3)
assert tuple(got[:2]) == (5, 5) and abs(got[2] - 8.0) < 1e-9, f"got {got}, expected (5, 5, 8.0): a one-piece answer has i = j"
print("Part 2 passed")

## Part 3 (coding task): Answer over a whole passage

Write `answer(question, passage, reader, max_len, overlap, max_answer_len)`. It returns
`(a, b)`, the first and last index *in the passage* of the best span over all windows.

- A window holds as many passage pieces as fit with the question in `max_len` positions.
- Consecutive windows share `overlap` passage pieces: if a window holds $m$ pieces, the
  next one starts $m - \text{overlap}$ pieces later. The first window starts at piece 0,
  and the last one is the first that reaches the end of the passage.
- Read each window with `pair_input`, `reader(input_ids, segment_ids, attention_mask)` and
  `best_span`, and keep the span with the highest score over all windows.

The check runs it with `max_len=48`, `overlap=12` and `max_answer_len=8`.

It also watches which windows your function hands to `reader`, so it can see the stride
and the window length and not only the indices you return.

In [ ]:
def answer(question, passage, reader, max_len, overlap, max_answer_len):
    """(a, b): passage indices of the best span over overlapping windows."""
    question = np.asarray(question)
    passage = np.asarray(passage)
    # TODO
    raise NotImplementedError

In [ ]:
window_counts = {}
for name, passage, expected in [("A", passage_a, (290, 292)), ("B", passage_b, (106, 109))]:
    seen = []

    def watching_reader(input_ids, segment_ids, attention_mask, seen=seen, name=name):
        try:
            held = [int(x) for x, s, a in zip(input_ids, segment_ids, attention_mask)
                    if int(s) == 1 and int(a) == 1 and int(x) != SEP_ID]
        except TypeError:
            raise AssertionError(
                f"passage {name}: the reader was handed {type(input_ids).__name__} instead of one window's "
                f"three arrays; read one window at a time, the way this part describes"
            )
        seen.append(held)
        return toy_reader(input_ids, segment_ids, attention_mask)

    got = tuple(int(x) for x in answer(question, passage, watching_reader, 48, 12, 8))
    assert got == expected, f"passage {name}: got {got}, expected {expected}" + (
        "; these look like positions in a window's input, not indices in the passage" if max(got) < 48 else ""
    )
    # The right answer can be reached by accident, because a grid of windows with the wrong
    # stride can still cover it. So read the windows back out of the reader and check them too.
    assert seen, f"passage {name}: the reader was never called, so no window was read"
    windows = []
    for held in seen:
        at = next((i for i in range(len(passage) - len(held) + 1)
                   if [int(x) for x in passage[i : i + len(held)]] == held), None)
        assert at is not None, f"passage {name}: a window held pieces that are not a run of this passage's pieces"
        windows.append((at, held))
    windows.sort()
    window_counts[name] = len(windows)
    room = 48 - len(question) - 3
    assert windows[0][0] == 0, (
        f"passage {name}: no window began at piece 0, the first began at piece {windows[0][0]}"
    )
    for at, held in windows:
        want = min(room, len(passage) - at)
        assert len(held) == want, (
            f"passage {name}: the window at piece {at} held {len(held)} pieces, expected {want}: a window holds "
            f"max_len - len(question) - 3 = {room} passage pieces, or however much of the passage is left"
        )
    for k, ((at, _), (later, _)) in enumerate(zip(windows, windows[1:])):
        assert later - at == room - 12, (
            f"passage {name}: window {k + 1} began at piece {later}, {later - at} pieces after window {k}, "
            f"which began at piece {at}; consecutive windows share overlap = 12 pieces, so each begins "
            f"room - overlap = {room - 12} pieces after the last"
        )
    covered = np.zeros(len(passage), dtype=int)
    for at, held in windows:
        covered[at : at + len(held)] += 1
    assert covered.min() > 0, (
        f"passage {name}: pieces {np.flatnonzero(covered == 0).tolist()[:6]} were in no window at all, so a span "
        f"there could never be found; the windows have to reach the passage's last piece"
    )
short = 3000 + np.arange(20)
short[[3, 5]] = [5001, 5002]
assert tuple(int(x) for x in answer(question, short, toy_reader, 48, 12, 8)) == (3, 5), (
    "a passage that fits in one window still needs its indices counted from the passage's first piece"
)
print("passage A: 300 pieces read in", window_counts["A"], "windows of 36, 24 pieces apart")
print("answer in passage A: pieces", answer(question, passage_a, toy_reader, 48, 12, 8))
print("answer in passage B: pieces", answer(question, passage_b, toy_reader, 48, 12, 8))
print("Part 3 passed")

## Solution

### Part 1: One question and passage as BERT's input

In [ ]:
def pair_input(question, passage, max_len):
    """input_ids, segment_ids and attention_mask for [CLS] question [SEP] passage [SEP], padded."""
    question = np.asarray(question)
    passage = np.asarray(passage)
    room = max_len - len(question) - 3              # [CLS] and two [SEP]s take three positions
    passage = passage[:room]
    real = np.concatenate([[CLS_ID], question, [SEP_ID], passage, [SEP_ID]])
    first_len = len(question) + 2                   # [CLS], the question and the first [SEP]
    input_ids = np.full(max_len, PAD_ID)
    segment_ids = np.zeros(max_len, dtype=int)
    attention_mask = np.zeros(max_len, dtype=int)
    input_ids[: len(real)] = real
    segment_ids[first_len : len(real)] = 1
    attention_mask[: len(real)] = 1
    return input_ids, segment_ids, attention_mask

**Why it works.** The three special tokens take three positions, so `max_len - len(question) - 3` pieces of the
passage fit, and slicing to that keeps the final `[SEP]` inside `max_len`. Cutting the
finished sequence at `max_len` would drop that `[SEP]` instead. Building the arrays full of
padding values and writing the real part over the front sets the padding's id, segment and
mask in one place. Segment 1 starts right after the first `[SEP]`, at position
`len(question) + 2`, and runs through the last `[SEP]`.

### Part 2: The best span in one input

In [ ]:
def best_span(start, end, input_ids, segment_ids, max_answer_len):
    """(i, j, score) of the best valid span: passage positions only, i <= j, at most max_answer_len pieces."""
    start = np.asarray(start, dtype=float)
    end = np.asarray(end, dtype=float)
    passage = (np.asarray(segment_ids) == 1) & (np.asarray(input_ids) != SEP_ID)
    t = np.arange(len(start))
    i, j = t[:, None], t[None, :]
    valid = passage[:, None] & passage[None, :] & (j >= i) & (j - i < max_answer_len)
    scores = np.where(valid, start[:, None] + end[None, :], -np.inf)
    a, b = np.unravel_index(np.argmax(scores), scores.shape)
    return int(a), int(b), float(scores[a, b])

**Why it works.** The best start and the best end, picked separately, need not form a span: in the first check
the best passage start is at 9 and the best end at 6. Scoring all pairs at once and
masking the invalid ones searches only real spans. A valid pair needs both ends in the
passage, which rules out the question, `[CLS]`, both `[SEP]`s and the padding, and it needs
$0 \le j - i < L$. The table has $T^2$ entries, about 150,000 for BERT's 384-token
reading inputs, which NumPy scores at once.

### Part 3: Answer over a whole passage

In [ ]:
def answer(question, passage, reader, max_len, overlap, max_answer_len):
    """(a, b): passage indices of the best span over overlapping windows."""
    question = np.asarray(question)
    passage = np.asarray(passage)
    room = max_len - len(question) - 3              # passage pieces per window
    offset = len(question) + 2                      # input position of a window's first passage piece
    top_score, found = -np.inf, (0, 0)
    first = 0                                       # passage index where the window begins
    while True:
        inputs = pair_input(question, passage[first : first + room], max_len)
        start, end = reader(*inputs)
        i, j, score = best_span(start, end, inputs[0], inputs[1], max_answer_len)
        if score > top_score:
            top_score, found = score, (first + i - offset, first + j - offset)
        if first + room >= len(passage):
            break
        first += room - overlap
    return found

**Why it works.** With 9 question pieces, a 48-position window holds $48 - 9 - 3 = 36$ passage pieces, and
windows start every $36 - 12 = 24$ pieces: at 0, 24, ..., 264 for a 300-piece passage, 12
windows in all. A window's first passage piece sits at input position
`len(question) + 2`, after `[CLS]`, the question and the first `[SEP]`, so a window starting
at passage piece `first` maps input position $i$ to passage index
$\text{first} + i - (\text{len(question)} + 2)$.

The overlap is what makes the search complete. Any span of at most $L$ pieces lies inside
some window when the overlap is at least $L - 1$, and here $12 \ge 7$. Passage B's answer,
pieces 106 to 109, crosses the boundary at 108 that windows of 36 without overlap would
have, and the window starting at 96 holds it whole. Passage A's answer, pieces 290 to 292,
lies past piece 275, so only the last window, which starts at 264, sees it. Each span is
scored the same way in every window that holds it, so comparing scores across windows is
fair.

The check reads the windows back out of the reader, so the stride is graded and not just
the answer. It has to be. The answer is the same passage index whichever window found it,
because a window beginning at `first` maps input position $i$ to passage index
$\text{first} + i - (\text{len(question)} + 2)$. So a grid of windows whose stride is wrong can
still cover the answer and hand back the same two indices, and a check that only looked at
those indices would pass it. Watching the windows is what makes the stride visible.